### Libraries

In [ ]:
import os
import re
import glob
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode, iplot
from datetime import datetime, timedelta
from scipy import ndimage, stats
from scipy.ndimage import center_of_mass
from io import BytesIO

# pyActigraphy
import pyActigraphy
from pyActigraphy.metrics.metrics import _lmx
from pyActigraphy.sleep.scoring_base import _td_format
from pyActigraphy.light.light import LightRecording



### Mask Application and Metric Computation

In [ ]:
warnings.filterwarnings('ignore')

# Load the mask database
maskdb = pd.read_csv('/home/jovyan/babyACTT/1.2_OffWrist_Manual/mascara_offwrist_manual_completo_oficial.csv')

# Helper function to extract the ID from the filename
def extract_id_from_filename(filename):
    match = re.search(r'P\d{2}[A-Za-z]\d{2}[A-Za-z]\d{1}_LogTAT', filename)
    return match.group(0) if match else None

# Helper function to extract the numerical ID from the mask database
def extract_numerical_id(id_value):
    match = re.search(r'(P\d{2}[A-Za-z]\d{2}[A-Za-z]\d{1}_LogTAT)', str(id_value))
    return match.group(0) if match else None

# Get a sorted list of filenames in the 'data' folder
directory_path = "/home/jovyan/babyACTT/1.1_OffWrist_Manualtxt/*LogTAT_conferido.txt"
filenames = sorted(glob.glob(directory_path, recursive=True))
print(len(filenames))

# Initialize an empty dictionary to store combined results
combined_filtered_data_dict = {}
# Process result
results = []

# Initialize the list to track mask adjustments
adjusted_files = []

for f in filenames:
    print(f)  # Print the current filename for debugging
    rawATR = pyActigraphy.io.read_raw_atr(f)   # Read the raw actigraphy data
    if isinstance(rawATR, RawAtr):
        pass
    file_id = extract_id_from_filename(f)
    print(f"Extracted file ID: {file_id}")
    starttime = rawATR.start_time
    print(f"Start time: {starttime}")  # Debug: print data start time

    # Replace light channel
    
    # Check available channels (optional, for debugging only)
    print(rawATR.light.get_channel_list())

    # Get data duration
    dur = rawATR.duration()
    print(f"Duration: {dur}")  # Debug: print duration

    # Calculate the end time based on start time and duration
    data_stop = starttime + dur
    print(f"Data range: Start = {starttime}, Stop = {data_stop}")

    # Counter for applied masks
    mask_applied_count = 0
    match_count = 0

    # Check all masks in the dataframe
    for index, row in maskdb.iterrows():
        db_id = extract_numerical_id(row['Subject_id'])

        if db_id == file_id:  # Check whether the mask belongs to this file
            start = row['Start_time']
            stop = row['Stop_time']
         # Adjust stop time to avoid exceeding the data limit
            stop = pd.Timestamp(stop)
            stop = min(stop, rawATR.data.index[-1])

            # Adjust stop time to avoid exceeding the data limit
            original_stop = stop  # Save original value
            stop = min(stop, rawATR.data.index[-1])

        # Check whether an adjustment occurred
            if stop != original_stop:
                adjusted_files.append({
                    'Filename': f,
                    'Original_Stop': original_stop,
                    'Adjusted_Stop': stop
                })

            # Apply the mask
            try:
                rawATR.add_mask_period(start=start, stop=stop)
                print(f"Mask applied: Start = {start}, Stop = {stop}")
                mask_applied_count += 1
                match_count += 1
            except ValueError as e:
                print(f"Error applying mask: {e}")
                # Add the mask period
            rawATR.add_mask_period(start=start, stop=stop)
            rawATR.light.create_light_mask()
            rawATR.light.add_light_mask_period(start=start, stop=stop)

            # Print the match
            print(f"Match found: Filename ID = {file_id}, Database ID = {db_id}, Start = {start}, Stop = {stop}")

    print(adjusted_files)
    print('adjusted_mask')
    for adjusted_file in adjusted_files:
        print(adjusted_file)
    print('final_adjusted_file')

    # Mask inactivity (needed to apply the mask)
    rawATR.mask_inactivity = True
    rawATR.ir_light.apply_mask = True

    # Compute the non-parametric variables
    RA = rawATR.RA(binarize=False)
    IV = rawATR.IV(binarize=False)
    IVm = rawATR.IVm(binarize=True)
    IS = rawATR.IS(binarize=False)
    ISm = rawATR.ISm(binarize=True)
    M10 = rawATR.M10(binarize=False)
    L5 = rawATR.L5(binarize=False)
    L5i = _lmx(rawATR.resampled_data('10min'), '5h', lowest=True)
    L5o = str(L5i[0])[7:]
    M10i = _lmx(rawATR.resampled_data('10min'), '10h', lowest=False)
    M10o = str(M10i[0])[7:]
    A_Offset = rawATR.AoffT(freq='10min', binarize=True)
    print(f"A_Offset: {A_Offset}")
    A_Onset = rawATR.AonT(freq='10min', binarize=True)
    print(f"A_Onset: {A_Onset}")

    # Light exposure (channels?, EDI, metrics - raw.light)
    L_Sum = rawATR.light.summary_statistics_per_time_bin()
    M10l = rawATR.light.LMX(length='10h', lowest=False)
    L5l = rawATR.light.LMX(length='5h', lowest=True)

    Lonset = L5l.iloc[0, 1]
    L5lo = str(Lonset)[7:]
    Monset = M10l.iloc[0, 1]
    M10lo = str(Monset)[7:]

    TAT_light = rawATR.ir_light.TAT(threshold=1, oformat='minute')
    IS_light = rawATR.ir_light.IS()
    IV_light = rawATR.ir_light.IV()

# --

    # ----------------------------------------
    lri_0 = rawATR.ir_light.LRI(threshold=0.25, get_profile=True)
    LRI = round(lri_0['IR LIGHT'], 4)

    # Light exposure (metrics - choosing the channel directly from the output) - IR

    # Centre of mass (gravity) + ADAT + Sleep fragmentation + SRI + SMP + FSoD + KRA
    daily_profile_cog = rawATR.average_daily_activity(freq='15min', cyclic=False, binarize=False)
    time_as_hours = daily_profile_cog.index.total_seconds() / 3600  # Convert seconds to hours
    Cog_timeashours = np.average(time_as_hours, weights=daily_profile_cog)

    daily_profile = rawATR.average_daily_activity(freq='15min', cyclic=False, binarize=False)
    CoG = ndimage.measurements.center_of_mass(daily_profile)  # To calculate the center of gravity or centroid (/2 if 60_)
    # CoG = ndimage.measurements.center_of_mass(daily_profile) # to calculate the center of gravity or centroid (/2 if 60_)
    print(f"CoG: {CoG}")

    daily_profile1 = rawATR.average_daily_activity(freq='10min', cyclic=True, binarize=True)
    AUC_total = (np.trapz(daily_profile1, axis=0) / 2)

    ADAT = rawATR.ADAT(binarize=True, threshold=4)
    pRA, pRA_weights = rawATR.pRA(4, start='00:00:00', period='8h')
    pRA2, pRA_weights2 = rawATR.pRA(4, start='18:00:00', period='12h')
    SMP = rawATR.SleepMidPoint()  # Sleep Midpoint - this one is being reported
    SMPtime = _td_format(SMP)
    FSoD = rawATR.fSoD(algo='Roenneberg', start='AonT', period='10h')  # Fraction of Sleep over Daytime
    KRA = rawATR.kRA(4, start='AoffT', freq='15min')  # Rest-to-Activity transition probability
    KAR = rawATR.kAR(0, start='AonT', freq='15min')

    # Calculate the SRI
    rawATR2 = rawATR
    rawATR2.data = rawATR.data.resample('30s').sum()
    SMP_CK = rawATR2.SleepMidPoint(freq='15min', bin_threshold=None, to_td=True, algo='CK')
    print(f"SMP_CK: {SMP_CK}")
    SRI = rawATR2.SleepRegularityIndex(freq='30min', algo='CK')

    # Reorganized dictionary: activity first, light second
    data = {
        # Activity variables
        'ID': file_id,
        'RA': RA,
        'IV': IV,
        'IVm': IVm,
        'IS': IS,
        'ISm': ISm,
        'M10': M10,
        'M10o': M10o,
        'L5': L5,
        'L5o': L5o,
        'SMPtime': SMPtime,
        'ADAT': ADAT,
        'CoG': CoG[0],
        'Cog_timeashours': Cog_timeashours,
        'AUC_total': AUC_total,
        'FSoD': FSoD,
        'KRA': KRA,
        'KAR': KAR,
        'SRI': SRI,
        'A_Onset': str(A_Onset),
        'A_Offset': str(A_Offset),
        #'SMP_CK': str(SMP_CK),
        #'pRA': pRA,
        #'pRA_weights': pRA_weights,

        # Light variables
        'LRI_light': LRI,  # Light Regularity Index
        'M10lo_light': M10lo,  # M10 Light Onset
        'L5lo_light': L5lo,  # L5 Light Onset
        # 'LRI_light_ir': LRI_ir,  # Light Regularity Index
        # 'M10lo_light_ir': M10lo_ir,  # M10 Light Onset
        # 'L5lo_light_ir': L5lo_ir,  # L5 Light Onset
        #'TAT_light': TAT_light,
        #'IS_light': IS_light,
        #'IV_light': IV_light,
    }
    results.append(data)

# Convert to a DataFrame
df = pd.DataFrame(results)

# Save the DataFrame to an Excel file
df.to_excel("Basic_extraction_oficial.xlsx", index=False)

print("Data saved to Basic_extraction_oficial.xlsx")

# Save the DataFrame to a CSV file
df.to_csv("Basic_extraction_oficial.csv", index=False)

print("Data saved to Basic_extraction_oficial.csv")

### Participant Group Identification

In [ ]:
# =============================================================================
# LOAD DATA
# =============================================================================
df = pd.read_excel("Basic_extraction_oficial.xlsx")


# =============================================================================
# EXTRACT METADATA FROM ID
# =============================================================================
df["Participant"] = df["ID"].str.extract(r"(P\d+)")
df["Sex"] = df["ID"].str.extract(r"([FM])")
df["Age"] = df["ID"].str.extract(r"(\d{2})(?=[RN])").astype(int)
df["Reactivity"] = df["ID"].str.extract(r"([RN])")
df["Diabetes"] = df["ID"].str.extract(r"(\d)_LogTAT").astype(int)


# =============================================================================
# REORDER COLUMNS
# =============================================================================
new_order = [
    "Participant", "Sex", "Age", "Reactivity", "Diabetes"
] + [
    col for col in df.columns
    if col not in ["Participant", "Sex", "Age", "Reactivity", "Diabetes"]
]

df = df[new_order]


# =============================================================================
# SAVE OUTPUT
# =============================================================================
df.to_excel("Basic_extraction_official_groups.xlsx", index=False)

print("Updated dataset saved as 'Basic_extraction_official_groups.xlsx'")

Dados atualizados e reorganizados em 'Basic_extraction_oficial_grupos.xlsx'


### Time Data Preprocessing

In [ ]:
# =============================================================================
# INPUT FILE
# =============================================================================
file_path = "Basic_extraction_oficial_grupos.xlsx"

# Read Excel file
df = pd.read_excel(file_path)


# =============================================================================
# FUNCTION: TIME STRING → MINUTES
# =============================================================================
def time_to_minutes(time_str):
    """
    Convert time strings like '0 days HH:MM:SS' or 'HH:MM:SS.sss'
    into minutes (float).
    """
    if pd.isna(time_str):
        return None

    if isinstance(time_str, str):
        # Remove '0 days ' prefix if present
        time_str = time_str.replace('0 days ', '')

        parts = time_str.split(':')

        if len(parts) == 3:
            hours, minutes, seconds = parts
            seconds = float(seconds)
            return int(hours) * 60 + int(minutes) + seconds / 60

    elif isinstance(time_str, (int, float)):
        # Already in minutes
        return time_str

    return None


# =============================================================================
# COLUMNS TO CONVERT
# =============================================================================
columns_to_convert = [
    'M10o', 'L5', 'L5o', 'SMPtime',
    'M10lo_light', 'L5lo_light',
    'A_Onset', 'A_Offset'
]

# Apply conversion
for col in columns_to_convert:
    df[col] = df[col].apply(time_to_minutes)


# =============================================================================
# CROSS-MIDNIGHT CORRECTION
# =============================================================================
# If values are very small (< 300 min = 5h), assume they occurred after midnight
# and adjust by adding 1440 min (24h)

for col in ['L5o', 'L5lo_light', 'A_Offset']:
    df[col] = df[col].apply(lambda x: x + 1440 if x is not None and x < 300 else x)


# =============================================================================
# DEBUG CHECK (OPTIONAL)
# =============================================================================
print(df['Cog_timeashours'].iloc[0])
print(type(df['Cog_timeashours'].iloc[0]))


# =============================================================================
# FUNCTION: EXTRACT NUMERIC FROM TUPLES
# =============================================================================
def extract_numeric(value):
    """
    Extract numeric value if the cell contains a tuple.
    """
    if isinstance(value, tuple):
        return value[0]
    return value


# Apply extraction
df['CoG'] = df['CoG'].apply(extract_numeric)
df['Cog_timeashours'] = df['Cog_timeashours'].apply(extract_numeric)


# =============================================================================
# VALIDATION CHECK
# =============================================================================
print(df[['CoG', 'Cog_timeashours']].head())


# =============================================================================
# SAVE OUTPUT
# =============================================================================
output_path = "Basic_extraction_oficial_grupos_modified_minutes.xlsx"
df.to_excel(output_path, index=False)

print(f"Modified file saved at: {output_path}")